In [1]:
import os
import random
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import transforms, datasets
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm

print("✅ Imports done!")

✅ Imports done!


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Using device: cuda
GPU: Tesla T4


In [3]:
SCENE_ROOT = "/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors"
TRAIN_DIR  = SCENE_ROOT + "/train"
TEST_DIR   = SCENE_ROOT + "/test"

print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

Train directory: /kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/train
Test directory: /kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/test


In [4]:
IMAGE_SIZE = 224        # ConvNeXt V2 native resolution
CROP_SIZE  = 256        # Oversample then crop → more variety
BATCH_SIZE = 16         # Use 8 if VRAM is tight with the Large model
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
 
train_transform = transforms.Compose([
    transforms.Resize((CROP_SIZE, CROP_SIZE)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=20),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2),
                             ratio=(0.3, 3.3), value=0),
])
 
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
 
print("✅ Transforms ready!")
IMAGE_SIZE = 224
BATCH_SIZE = 32

# ConvNeXt works best with strong augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("✅ Transforms ready!")

✅ Transforms ready!
✅ Transforms ready!


In [5]:
base_dataset = datasets.ImageFolder(root=TRAIN_DIR)
class_names  = base_dataset.classes
num_classes  = len(class_names)

print("Classes found:", class_names)
print("Total images:", len(base_dataset))
print("Number of classes:", num_classes)

indices = np.arange(len(base_dataset))
targets = np.array(base_dataset.targets)

train_indices, val_indices = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=targets
)

train_full    = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_full      = datasets.ImageFolder(root=TRAIN_DIR, transform=val_transform)
train_dataset = Subset(train_full, train_indices)
val_dataset   = Subset(val_full,   val_indices)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Classes found: ['asian', 'boho', 'coastal', 'contemporary', 'craftsman', 'eclectic', 'farmhouse', 'french-country', 'industrial', 'mediterranean', 'minimalist', 'modern', 'scandinavian', 'shabby-chic-style', 'southwestern', 'tropical', 'victorian']
Total images: 13163
Number of classes: 17
Training images: 10530
Validation images: 2633


In [6]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 330
Validation batches: 83


In [7]:
class SceneClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # ConvNeXt-Base — strongest model that fits in T4 GPU memory
        convnext = models.convnext_base(
            weights=models.ConvNeXt_Base_Weights.DEFAULT
        )

        # Professor's structure
        self.features = convnext.features  # all conv blocks
        self.avgpool  = nn.AdaptiveAvgPool2d((1, 1))

        # ConvNeXt outputs 1024 features
        in_features = 1024
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(in_features),      # ConvNeXt uses LayerNorm not BatchNorm
            nn.Linear(in_features, 512),
            nn.GELU(),                      # ConvNeXt uses GELU not ReLU
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

    def freeze_backbone(self):
        for param in self.features.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        # Unfreeze last 3 stages of ConvNeXt
        for param in self.features[5:].parameters():
            param.requires_grad = True


model = SceneClassifier(num_classes=num_classes).to(device)
model.freeze_backbone()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Stage 1 — Trainable: {trainable:,}  |  Frozen: {frozen:,}")

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:01<00:00, 183MB/s]


Stage 1 — Trainable: 535,569  |  Frozen: 87,564,416


In [8]:
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index   = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def cutmix_data(x, y, alpha=1.0):
    lam   = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    W, H  = x.size(3), x.size(2)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx    = np.random.randint(W)
    cy    = np.random.randint(H)
    x1    = max(cx - cut_w // 2, 0)
    y1    = max(cy - cut_h // 2, 0)
    x2    = min(cx + cut_w // 2, W)
    y2    = min(cy + cut_h // 2, H)
    mixed_x         = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed_x, y, y[index], lam

def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("✅ MixUp + CutMix ready!")

✅ MixUp + CutMix ready!


In [9]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=0.05  # ConvNeXt benefits from higher weight decay
)

scheduler_s1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

print("✅ Optimizer ready!")

✅ Optimizer ready!


In [10]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, use_mix=True):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for images, labels in tqdm(dataloader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        if use_mix:
            # Randomly choose MixUp or CutMix each batch
            if random.random() < 0.5:
                images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.4)
            else:
                images, y_a, y_b, lam = cutmix_data(images, labels, alpha=1.0)
            outputs = model(images)
            loss    = mixed_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            outputs = model(images)
            loss    = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        predicted   = outputs.argmax(dim=1)
        correct    += (predicted == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct    = 0
    total      = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)

            outputs    = model(images)
            loss       = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            predicted   = outputs.argmax(dim=1)
            correct    += (predicted == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

print("✅ Functions ready!")

✅ Functions ready!


In [11]:
STAGE1_EPOCHS = 10
STAGE2_EPOCHS = 20
TOTAL_EPOCHS  = STAGE1_EPOCHS + STAGE2_EPOCHS

best_val_acc     = 0.0
best_model_state = None
train_losses, val_losses         = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(TOTAL_EPOCHS):

    if epoch == STAGE1_EPOCHS:
        print("\n" + "="*60)
        print("SWITCHING TO STAGE 2: Unfreezing last 3 ConvNeXt stages")
        print("="*60 + "\n")

        model.unfreeze_backbone()

        optimizer = torch.optim.AdamW([
            {"params": model.classifier.parameters(),   "lr": 1e-4},
            {"params": model.features[5:].parameters(), "lr": 1e-5},
        ], weight_decay=0.05)

        scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=STAGE2_EPOCHS, eta_min=1e-7
        )

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Stage 2 — Trainable: {trainable:,} parameters")

    # MixUp + CutMix in both stages
    use_mix    = True
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, use_mix
    )
    val_loss, val_acc, val_preds, val_labels_arr = evaluate(
        model, val_loader, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        best_model_state = model.state_dict()
        torch.save(best_model_state, "/kaggle/working/best_model.pth")
        print("✅ Best model saved!")

    if epoch < STAGE1_EPOCHS:
        scheduler_s1.step(val_acc)
    else:
        scheduler_s2.step()

    stage = "S1" if epoch < STAGE1_EPOCHS else "S2"
    print(
        f"[{stage}] Epoch [{epoch+1}/{TOTAL_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

Evaluating: 100%|██████████| 83/83 [00:34<00:00,  2.41it/s]


✅ Best model saved!
[S1] Epoch [1/30] Train Loss: 2.5465 | Train Acc: 0.1961 | Val Loss: 2.2109 | Val Acc: 0.3403


Evaluating: 100%|██████████| 83/83 [00:33<00:00,  2.51it/s]


✅ Best model saved!
[S1] Epoch [2/30] Train Loss: 2.4527 | Train Acc: 0.2237 | Val Loss: 2.1271 | Val Acc: 0.3794


Evaluating: 100%|██████████| 83/83 [00:33<00:00,  2.48it/s]


...

✅ Best model saved!
[S2] Epoch [28/30] Train Loss: 2.0685 | Train Acc: 0.3673 | Val Loss: 1.8756 | Val Acc: 0.4789


Evaluating: 100%|██████████| 83/83 [00:32<00:00,  2.52it/s]


✅ Best model saved!
[S2] Epoch [29/30] Train Loss: 2.0768 | Train Acc: 0.3880 | Val Loss: 1.8742 | Val Acc: 0.4804


Evaluating: 100%|██████████| 83/83 [00:32<00:00,  2.52it/s]

[S2] Epoch [30/30] Train Loss: 2.0732 | Train Acc: 0.3662 | Val Loss: 1.8744 | Val Acc: 0.4801

Best validation accuracy: 0.4804


In [12]:
model.load_state_dict(best_model_state)
model.to(device)

_, final_acc, val_preds, val_labels_arr = evaluate(
    model, val_loader, criterion, device
)
print("Final Validation Accuracy:", final_acc)
print()
print(classification_report(val_labels_arr, val_preds, target_names=class_names))

Evaluating: 100%|██████████| 83/83 [00:33<00:00,  2.51it/s]

Final Validation Accuracy: 0.4800607671857197

                   precision    recall  f1-score   support

            asian       0.53      0.44      0.48       156
             boho       0.83      0.89      0.86       184
          coastal       0.37      0.43      0.40       159
     contemporary       0.26      0.26      0.26       156
        craftsman       0.39      0.48      0.43       153
         eclectic       0.59      0.42      0.49       162
        farmhouse       0.31      0.42      0.36       159
   french-country       0.37      0.41      0.39       158
       industrial       0.65      0.50      0.57       153
    mediterranean       0.45      0.37      0.41       158
       minimalist       0.75      0.80      0.77       111
           modern       0.38      0.51      0.44       162
     scandinavian       0.40      0.47      0.43       153
shabby-chic-style       0.57      0.35      0.43       149
     southwestern       0.48      0.51      0.49       154
        

In [13]:
class TestImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.transform   = transform
        self.image_paths = image_paths
        print(f"Total test images: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        file_name = os.path.basename(self.image_paths[index])
        try:
            image = Image.open(self.image_paths[index]).convert("RGB")
        except Exception:
            print(f"Corrupted image replaced with blank: {file_name}")
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, file_name

In [14]:
IMAGE_EXTENSIONS = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
test_image_paths = []

for ext in IMAGE_EXTENSIONS:
    test_image_paths.extend(glob.glob(os.path.join(TEST_DIR, ext)))

test_image_paths = sorted(test_image_paths)
print("Test images found:", len(test_image_paths))

test_dataset = TestImageDataset(test_image_paths, transform=val_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Test images found: 5482
Total test images: 5482


In [15]:
# Load best model
model.load_state_dict(torch.load("/kaggle/working/best_model.pth",
                                  map_location=device))
model.to(device)
model.eval()
print("✅ Best model loaded!")

# 7 TTA transforms
tta_transforms = [
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE + 64, IMAGE_SIZE + 64)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
]

all_filenames   = []
all_predictions = []

with torch.no_grad():
    for img_path in tqdm(test_image_paths, desc="Predicting with TTA"):
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))

        file_name  = os.path.basename(img_path)
        logits_sum = None

        for tta_t in tta_transforms:
            img_tensor = tta_t(image).unsqueeze(0).to(device)
            output     = model(img_tensor)
            if logits_sum is None:
                logits_sum = output
            else:
                logits_sum += output

        pred = logits_sum.argmax(dim=1).item()
        all_filenames.append(file_name)
        all_predictions.append(pred)

submission = pd.DataFrame({
    "ImageName": all_filenames,
    "label":     all_predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("✅ submission.csv saved!")
print(f"Total rows: {len(submission)}")
print(submission.head(10))

✅ Best model loaded!


Predicting with TTA:  49%|████▉     | 2682/5482 [06:00<05:35,  8.35it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Predicting with TTA: 100%|██████████| 5482/5482 [12:12<00:00,  7.48it/s]

✅ submission.csv saved!
Total rows: 5482
            ImageName  label
0     testimage_1.jpg     12
1    testimage_10.jpg      7
2   testimage_100.jpg     10
3  testimage_1000.jpg      2
4  testimage_1001.jpg     10
5  testimage_1002.jpg     10
6  testimage_1003.jpg     16
7  testimage_1004.jpg      0
8  testimage_1005.jpg     10
9  testimage_1006.jpg     16
